Practice notebook for a complete pytorch workflow; dummy implementation of LinearRegression model using pytorch 

In [1]:
import sys, os
import torch
from torch import nn
import matplotlib.pyplot as plt

print(torch.__version__)

2.10.0+cpu


In [2]:
device = "cuda" if  torch.cuda.is_available() else "cpu"
print(f'Device: {device}')


Device: cpu


In [3]:
root = os.path.abspath(os.path.join(os.getcwd(), ".."))

sys.path.insert(0, root)

In [4]:
import pandas as pd
import numpy as np

data = pd.read_csv("../data/BTC_USDT_5m.csv", parse_dates=["open_time"],index_col="open_time", dtype=np.float64)
data.head() 

,open,high,low,close,volume
open_time,,,,,
2017-12-31 00:00:00,12345.10,12397.16,12287.89,12311.02,36.479616
2017-12-31 00:05:00,12313.00,12410.46,12270.74,12270.74,56.512808
2017-12-31 00:10:00,12270.74,12300.12,12149.98,12211.95,75.285130
2017-12-31 00:15:00,12206.82,12489.05,12201.27,12391.00,70.083180
2017-12-31 00:20:00,12379.92,12489.04,12296.28,12468.41,87.112825


In [5]:
import sklearn
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

In [6]:
## Splitting data
train_size = int(0.8 * len(data))
print(f'Data size: {data.shape}')
print(f'Training sample: {data.shape[0]}')

X = data.drop("close", axis=1)
print(f'features: {X.columns}')

y = data['close']
print(f'target: {y.name}')



Data size: (762998, 5)
Training sample: 762998
features: Index(['open', 'high', 'low', 'volume'], dtype='object')
target: close


In [7]:
X_train, y_train, X_test, y_test = X[:train_size], y[:train_size], X[train_size:], y[train_size:]

print(f'Training samples: {len(X_train)}')


Training samples: 610398


In [8]:
X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)



In [9]:
print(X_train_scaled.dtype)
print(X_train_scaled)

float64
[[-0.55519013 -0.55295743 -0.55772201 -0.54521982]
 [-0.55720478 -0.55212399 -0.55880005 -0.49932459]
 [-0.5598571  -0.55903842 -0.566391   -0.45631797]
 ...
 [ 0.83942692  0.83824419  0.84219726 -0.46158836]
 [ 0.83920223  0.83624895  0.84070057 -0.50258217]
 [ 0.83943571  0.83690693  0.84115379 -0.52605   ]]


In [10]:
print('Data to tensors:....')
X_train = torch.tensor(X_train_scaled, dtype=torch.float, device=device)
y_train = torch.tensor(y_train.values, dtype=torch.float, device=device)

Data to tensors:....


In [11]:
X_train.shape, y_train.shape

(torch.Size([610398, 4]), torch.Size([610398]))

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(X_train, y_train)

dataloader = DataLoader(dataset=dataset,
                        batch_size=64,
                        shuffle=False,
                        num_workers=4)
 

for batch_x, batch_y in dataloader:
    print(f'Batch X Shape: {batch_x.shape}')
    print(f'Batch y Shape: {batch_y.shape}')



Batch X Shape: torch.Size([64, 4])
Batch y Shape: torch.Size([64])
Batch X Shape: torch.Size([64, 4])
Batch y Shape: torch.Size([64])
Batch X Shape: torch.Size([64, 4])
Batch y Shape: torch.Size([64])
Batch X Shape: torch.Size([64, 4])
Batch y Shape: torch.Size([64])
Batch X Shape: torch.Size([64, 4])
Batch y Shape: torch.Size([64])
Batch X Shape: torch.Size([64, 4])
Batch y Shape: torch.Size([64])
Batch X Shape: torch.Size([64, 4])
Batch y Shape: torch.Size([64])
Batch X Shape: torch.Size([64, 4])
Batch y Shape: torch.Size([64])
Batch X Shape: torch.Size([64, 4])
Batch y Shape: torch.Size([64])
Batch X Shape: torch.Size([64, 4])
Batch y Shape: torch.Size([64])
Batch X Shape: torch.Size([64, 4])
Batch y Shape: torch.Size([64])
Batch X Shape: torch.Size([64, 4])
Batch y Shape: torch.Size([64])
Batch X Shape: torch.Size([64, 4])
Batch y Shape: torch.Size([64])
Batch X Shape: torch.Size([64, 4])
Batch y Shape: torch.Size([64])
Batch X Shape: torch.Size([64, 4])
Batch y Shape: torch.Size([

In [23]:
print(batch_x[0])

batch_x, batch_y = next(iter(dataloader))

print(batch_x.shape)


tensor([ 0.8420,  0.8415,  0.8438, -0.4169])
torch.Size([64, 4])


#### Model

In [24]:
## Model
class LinearRegression(nn.Module):
    def __init__(self, features: torch.Tensor, output=1):
        super().__init__()
        self.linear=nn.Linear(features.shape[1], output)
        
    def forward(self, X: torch.Tensor) -> torch.Tensor:
        return self.linear(X)
    


In [25]:
print(LinearRegression.__mro__)

(<class '__main__.LinearRegression'>, <class 'torch.nn.modules.module.Module'>, <class 'object'>)


In [26]:
torch.manual_seed(42)
model = LinearRegression(X_train)

print(f'Model Parameters: {model.state_dict()}')
print(f"Weights: {model.state_dict()['linear.weight']}")
print(f'Bias: {model.state_dict()["linear.bias"]}')

Model Parameters: OrderedDict([('linear.weight', tensor([[ 0.3823,  0.4150, -0.1171,  0.4593]])), ('linear.bias', tensor([-0.1096]))])
Weights: tensor([[ 0.3823,  0.4150, -0.1171,  0.4593]])
Bias: tensor([-0.1096])


In [27]:
# Checking current model device
next(model.parameters()).device

device(type='cpu')

In [28]:
model.to(device)
next(model.parameters()).device

device(type='cpu')

#### Training

In [29]:
loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)



In [30]:
## Training loop
torch.manual_seed(42)
epochs = 200

for epoch in range(epochs):
    #
    model.train()

    for batch_x, batch_y in dataloader:

        # forward pass
        preds = model(batch_x)

        # calculate loss
        loss = loss_fn(preds, batch_y)
        

        optimizer.zero_grad()

        loss.backward()
        
        optimizer.step()

    model.eval()

        
    if epoch % 20 == 0:
        print(f'Epoch: {epoch} | Training Loss: {loss} | ')


Epoch: 0 | Training Loss: 1736793.375 | 
Epoch: 20 | Training Loss: 10626.650390625 | 
Epoch: 40 | Training Loss: 10619.291015625 | 
Epoch: 60 | Training Loss: 10612.732421875 | 
Epoch: 80 | Training Loss: 10606.193359375 | 
Epoch: 100 | Training Loss: 10599.8876953125 | 
Epoch: 120 | Training Loss: 10592.80859375 | 
Epoch: 140 | Training Loss: 10588.0107421875 | 
Epoch: 160 | Training Loss: 10581.4931640625 | 
Epoch: 180 | Training Loss: 10576.853515625 | 


In [31]:
X_test_scaled = torch.tensor(X_test_scaled, dtype=torch.float, device=device)
y_test = torch.tensor(y_test, dtype=torch.float, device=device)


y_test.dtype

C:\Users\user\AppData\Local\Temp\ipykernel_8708\2491590831.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  y_test = torch.tensor(y_test, dtype=torch.float, device=device)


torch.float32

In [32]:
model.eval()
with torch.inference_mode():
    test_pred = model(X_test_scaled[:3000])
    test_loss = loss_fn(test_pred, y_test[:3000])

test_pred

c:\Users\user\Documents\dev\deeplearning\venv\lib\site-packages\torch\nn\modules\loss.py:626: UserWarning: Using a target size (torch.Size([3000])) that is different to the input size (torch.Size([3000, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


tensor([[34465.2422],
        [34465.1641],
        [34581.1641],
        ...,
        [35022.2227],
        [35048.8398],
        [35046.5820]])

In [33]:
test_loss 
## Conclusion: overfitted the training data

tensor(313901.7812)

#### Saving Model

In [34]:
## Saving model

from pathlib import Path

MODEL_PATH = Path('models')
MODEL_PATH.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "01_LINEAR_MODEL_.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

print(f'saving model to path: {MODEL_SAVE_PATH}')
torch.save(obj=model.state_dict(), f=MODEL_SAVE_PATH)


saving model to path: models\01_LINEAR_MODEL_.pth


#### Loading saved model

In [35]:
## Instantiate Model
load_model = LinearRegression(X_train_scaled)

## load model saved state
print(f'Loading model from path: {MODEL_SAVE_PATH}')
load_model.load_state_dict(torch.load(f=MODEL_SAVE_PATH))

print(load_model.state_dict())

Loading model from path: models\01_LINEAR_MODEL_.pth
OrderedDict([('linear.weight', tensor([[5126.7490, 6121.1294, 4532.2764,   -7.7568]])), ('linear.bias', tensor([21276.9453]))])


In [36]:
## Testing model
load_model.eval()
with torch.inference_mode():
    preds = load_model(X_test_scaled[:3000])
    loss = loss_fn(preds, y_test[:3000])

loss

tensor(313901.7812)